# DataMate pipeline — chạy trên Kaggle (GPU T4 ×2), code lấy từ GitHub

**Chuẩn bị (1 lần):**
1. Đẩy repo lên GitHub (nên để **private**). Sửa `REPO_URL` ở bước 3.
2. Repo private: tạo GitHub token chỉ đọc → Kaggle *Add-ons → Secrets* → thêm secret tên `GITHUB_TOKEN`.
3. *Add Input* dataset dữ liệu (mặc định `/kaggle/input/datasets/gmnguynhng/copypaste`; notebook tự tìm nếu đường dẫn khác). Dataset có thể là thư mục đã giải nén hoặc file `.zip`.
4. *Settings → Accelerator = GPU T4 x2*; *Internet = On* (để clone repo, cài vLLM và tải model).

**Thứ tự chạy:** 1 → 2 → 3 → 4 → 5 (đo nhanh) → 6 (đánh giá trên TRAIN, có điểm) → 7 (chạy PUBLIC, ra file nộp).

> **Chỉ chấm điểm trên TRAIN.** Notebook không so kết quả public với bất kỳ file đáp án public nào (quy định 6: không thu thập/dùng đáp án câu hỏi test). Điểm public xem trên leaderboard sau khi nộp.


## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch, sys
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| số GPU", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))

## 2. Cài thư viện
vLLM nhanh hơn nhiều (batching, prefix caching). Nếu cài vLLM lỗi trên T4, dùng backend `hf` (transformers) — chậm hơn nhưng chạy được.

In [ ]:
%%capture install_log
!pip install -q "vllm>=0.6.3" python-calamine


In [ ]:
# Kiểm vLLM trong TIẾN TRÌNH MỚI (kernel này đã import torch cũ, import vllm ở đây có thể báo lỗi giả)
import subprocess
r = subprocess.run(["python", "-c", "import vllm, torch; print(vllm.__version__, torch.__version__)"], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])
LLM_BACKEND = "vllm" if r.returncode == 0 else "hf"
if LLM_BACKEND == "hf":
    subprocess.run("pip install -q -U transformers accelerate bitsandbytes autoawq python-calamine", shell=True)
print("BACKEND =", LLM_BACKEND)


## 3. Clone code từ GitHub + tìm dữ liệu

In [ ]:
import os, glob, shutil, zipfile
WORK = "/kaggle/working"
DATA = f"{WORK}/data"
os.makedirs(DATA, exist_ok=True)
INPUT_DIR = "/kaggle/input/datasets/gmnguynhng/copypaste"     # ← đường dẫn dataset của bạn
if not os.path.exists(INPUT_DIR):
    print("Không thấy", INPUT_DIR, "→ tìm trong toàn bộ /kaggle/input")
    INPUT_DIR = "/kaggle/input"
!ls -R "$INPUT_DIR" | head -40

# giải nén nếu dataset còn ở dạng .zip
for z in glob.glob(f"{INPUT_DIR}/**/*.zip", recursive=True):
    print("giải nén", z)
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA)
for z in glob.glob(f"{DATA}/**/*.zip", recursive=True):   # zip lồng nhau (DATAMATE.zip chứa train.zip, public_test.zip)
    with zipfile.ZipFile(z) as f:
        f.extractall(os.path.dirname(z))

def find_split(name):
    for root in [INPUT_DIR, DATA]:
        for p in glob.glob(f"{root}/**/rulebook.md", recursive=True):
            d = os.path.dirname(p)
            if name in d.lower() and os.path.exists(f"{d}/questions.jsonl"):
                return d
    return None

TRAIN = find_split("train")
PUBLIC = find_split("public")
print("TRAIN :", TRAIN)
print("PUBLIC:", PUBLIC)
assert TRAIN and PUBLIC, "Không tìm thấy thư mục train/public (cần có rulebook.md + questions.jsonl). Xem cây thư mục in ở trên."
for d in (TRAIN, PUBLIC):
    print(d, "| số shop:", len(os.listdir(f"{d}/data")), "| số câu:", sum(1 for _ in open(f"{d}/questions.jsonl", encoding="utf-8")))

# ---- clone code ----
REPO_URL = "https://github.com/<ten-ban>/datamate-pipeline.git"   # ← sửa
BRANCH = "main"
CODE = f"{WORK}/code"
url = REPO_URL
try:   # repo private: dùng token trong Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@")
except Exception:
    print("Không có secret GITHUB_TOKEN → clone như repo public")
if os.path.exists(CODE):
    shutil.rmtree(CODE)
rc = os.system(f"git clone -q --depth 1 -b {BRANCH} {url} {CODE}")
assert rc == 0, "clone lỗi: kiểm tra REPO_URL / token"
os.system(f"git -C {CODE} remote set-url origin {REPO_URL}")   # xoá token khỏi .git/config
os.system(f"git -C {CODE} log -1 --oneline")   # ghi lại commit đang chạy (tái lập)
os.chdir(CODE)
print(sorted(os.listdir(CODE)))

## 4. Self-test (parser, thời gian, chấm điểm, sandbox) — phải ra `SELFTEST: OK`

In [ ]:
!python datamate_pipeline.py selftest --split_dir "$TRAIN"

In [ ]:
# Kiểm bộ bảng chuẩn V + hàm nghiệp vụ bằng đáp án TRAIN (không cần GPU, ~10 phút)
!python scripts/validate_canonical.py --train "$TRAIN" --show_bad
!python scripts/validate_semantics.py --train "$TRAIN"
!python scripts/eval_compiler.py --train "$TRAIN" --norm --show_bad 20   # đường máy tính: độ phủ, precision, PA (chỉ train)


In [ ]:
# Xem thẻ dữ liệu mà model sẽ đọc của 1 shop
!python datamate_pipeline.py card --split_dir "$TRAIN" --biz biz_101

## 5. Chọn model + đo nhanh (2 shop × 8 câu)
- `Qwen/Qwen2.5-Coder-7B-Instruct-AWQ`: ~5,5 GB, viết code tốt (mặc định). `TP=2` chia qua 2 T4 → gấp đôi bộ nhớ cho KV cache, chạy lô lớn hơn.
- Nếu vLLM lỗi khi `TP=2`: đặt `TP=1`.
- Muốn thử nhanh hơn: `Qwen/Qwen2.5-3B-Instruct-AWQ`.

Kiểm tra lại **ngày phát hành và giấy phép** của model trước khi dùng cho bản nộp.


In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
TP = 2                 # 2 T4; nếu lỗi thì 1
MAX_LEN = 28000        # rulebook ~9k token + thẻ dữ liệu + bộ bảng chuẩn V + câu hỏi (dài nhất ~17k token); prompt quá dài tự bị bỏ qua (abstain), không làm hỏng cả lô
N_PATHS = 2            # 2 đường độc lập, chỉ trả lời khi đồng thuận
BATCH = 48 if LLM_BACKEND == "vllm" else 4
LM_TEXTS = f"{TRAIN}/questions.jsonl"   # bộ sửa chữ học thêm từ câu hỏi TRAIN (hợp lệ: chỉ train)
COMMON = (f"--llm {LLM_BACKEND} --model {MODEL} --tp {TP} --max_model_len {MAX_LEN} --n_paths {N_PATHS} "
          f"--batch {BATCH} --cache {WORK}/cache --lm_texts {LM_TEXTS}")
print(COMMON)


In [ ]:
import time
t = time.time()
!python datamate_pipeline.py run --split_dir "$TRAIN" --out "$WORK/out_smoke" $COMMON --biz biz_101,biz_119 --limit 8 --fresh
print(f"thời gian: {time.time()-t:.0f}s")
!python datamate_pipeline.py score --split_dir "$TRAIN" --pred "$WORK/out_smoke/submission.jsonl" --only_pred

## 5b. (Tuỳ chọn) Đo LLM sửa chữ trên câu TRAIN làm bẩn
So 3 cách: chỉ luật + mô hình ngôn ngữ / thêm LLM chọn trắc nghiệm ở chỗ chưa chắc (mặc định pipeline, `--norm_llm main`) / LLM tự viết lại cả câu.
Nếu `llm_chon` không hơn `ngu_canh` ở mọi dòng (nhất là dòng `sach`), đặt `--norm_llm off` trong `COMMON`.
Muốn thử model tiếng Việt: đổi `--llm hf --model vinai/PhoGPT-4B-Chat` (kiểm giấy phép trước khi dùng cho bản nộp).


In [ ]:
!python scripts/eval_norm_llm.py --train "$TRAIN" --llm {LLM_BACKEND} --model {MODEL} --tp {TP} --n 300


## 5c. (Tuỳ chọn) PhoGPT + bge-m3 — đo trước khi bật
Thêm dataset Kaggle chứa thư mục model offline rồi điền đường dẫn. Tổng tham số Qwen 7,6B + PhoGPT 3,7B + bge-m3 0,57B ≈ 11,9B.
- **PhoGPT** chỉ CHẤM các câu ứng viên ở chỗ sửa chữ chưa chắc (không viết lại câu → không làm hỏng số / tên hàng). Bật nếu cột `ca_hai` hoặc `pho_cham` cao hơn `llm_chon` mà câu SẠCH vẫn 100%.
- **bge-m3** tìm câu train giống nhất làm ví dụ mẫu cho LLM viết đặc tả. Bật nếu dòng `ví dụ mẫu=có` có tỉ lệ đúng cao hơn và SAI không tăng.
- T4 16 GB: PhoGPT chạy pha riêng TRƯỚC vLLM rồi giải phóng GPU, nên không phải chia bộ nhớ với Qwen.

In [ ]:
PHO_DIR = ""     # vd "/kaggle/input/phogpt-4b-chat/PhoGPT-4B-Chat" — trống = bỏ qua
BGE_DIR = ""     # vd "/kaggle/input/bge-m3/bge-m3"
!pip install -q "transformers>=4.44" sentence-transformers
if PHO_DIR:
    !python scripts/eval_norm_llm.py --train "$TRAIN" --llm {LLM_BACKEND} --model {MODEL} --tp {TP} --n 300 --no_rewrite --pho_model "$PHO_DIR"
!python scripts/eval_llm_spec.py --train "$TRAIN" --llm {LLM_BACKEND} --model {MODEL} --tp {TP} --n 400 {"--embed_model " + BGE_DIR if BGE_DIR else ""}
EXTRA = (f" --norm_llm both --pho_model {PHO_DIR}" if PHO_DIR else "") + (f" --embed_model {BGE_DIR}" if BGE_DIR else "")
COMMON = COMMON + EXTRA
print(COMMON)
!python scripts/eval_ask_choice.py --train "$TRAIN" --llm {LLM_BACKEND} --model {MODEL} --tp {TP} --n 300   # A/B/C: bật --ask_choice nếu đúng > sai rõ


## 6. Đánh giá trên TRAIN (đây là bảng điểm để phát triển)
Chạy trên một nhóm shop train làm "shop mù" (không nhìn khi sửa prompt/luật). Muốn chạy toàn bộ 30 shop thì bỏ `--biz` (lâu hơn).

In [ ]:
BLIND = "biz_103,biz_108,biz_113,biz_118,biz_124,biz_129"   # ví dụ 6 shop mù; giữ cố định suốt quá trình phát triển
t = time.time()
!python datamate_pipeline.py run --split_dir "$TRAIN" --out "$WORK/out_train" $COMMON --biz $BLIND --time_budget 30000
print(f"thời gian: {(time.time()-t)/60:.1f} phút")
!python datamate_pipeline.py score --split_dir "$TRAIN" --pred "$WORK/out_train/submission.jsonl" --only_pred

### 6b. Hiệu chỉnh bỏ trống theo PA (từ log TRAIN vừa chạy)
Nhãn quyết định có tỉ lệ đúng ≤ 0,5 trên train sẽ bị đổi thành bỏ trống khi chạy public/private. Chỉ dùng train.

In [ ]:
!python scripts/calibrate_decisions.py --train "$TRAIN" --run "$WORK/out_train" --out lexicon/decision_calibration.json
!python scripts/fit_abstain.py --train "$TRAIN" --run "$WORK/out_train" --out lexicon/abstain_model.json
!cat "$WORK/out_train/run_summary.json"   # số lượt gọi model mỗi câu, giây mỗi câu, bước đã tự tắt
COMMON = COMMON + " --abstain_model lexicon/abstain_model.json --calibration lexicon/decision_calibration.json"
print(COMMON)


In [ ]:
# Xem các câu sai + log để tìm loại lỗi
import json, pandas as pd
errs = [json.loads(l) for l in open(f"{WORK}/out_train/errors.jsonl", encoding="utf-8")]
df = pd.DataFrame([{"id": e["id"], "shop": e["business_id"], "câu hỏi": e["question"], "gold": e["gold"]["answer"],
                    "dự đoán": (e["pred"] or {}).get("answer")} for e in errs])
pd.set_option("display.max_colwidth", 120)
df.head(30)

In [ ]:
# Xem chi tiết 1 câu: cảnh báo, kỳ thời gian, code của 2 đường, quyết định
qid = df.id.iloc[0] if len(df) else None
if qid:
    log = json.load(open(f"{WORK}/out_train/logs/{qid}.json", encoding="utf-8"))
    print("CÂU:", log["question"]); print("CHUẨN HOÁ:", log["normalized"]); print("CẢNH BÁO:", log["warnings"])
    print("QUYẾT ĐỊNH:", log["decision"], log["final"])
    for k, v in log["paths"].items():
        print(f"\n===== ĐƯỜNG {k} | lỗi: {v['error']} | kết quả: {v['line']}\n{v['code']}")

## 7. Chạy PUBLIC TEST → file nộp
Ra `out_public/submission.jsonl`. Không chấm ở đây: nộp file lên leaderboard để xem điểm public.


In [ ]:
t = time.time()
!python datamate_pipeline.py run --split_dir "$PUBLIC" --out "$WORK/out_public" $COMMON --time_budget 39000
print(f"thời gian: {(time.time()-t)/60:.1f} phút")
!wc -l "$WORK/out_public/submission.jsonl"; cat "$WORK/out_public/run_summary.json"


In [ ]:
# Kiểm format file nộp: đủ câu, không trùng id, type hợp lệ (không cần đáp án)
import collections
qs = [json.loads(l)["id"] for l in open(f"{PUBLIC}/questions.jsonl", encoding="utf-8")]
rows = [json.loads(l) for l in open(f"{WORK}/out_public/submission.jsonl", encoding="utf-8")]
dup = [k for k, v in collections.Counter(r["id"] for r in rows).items() if v > 1]
sub = {r["id"]: r for r in rows}
missing = [q for q in qs if q not in sub]
extra = [k for k in sub if k not in set(qs)]
print("số câu:", len(qs), "| đã có:", len(sub), "| thiếu (tính abstain):", len(missing), "| trùng id:", len(dup), "| id lạ:", len(extra))
print(pd.Series([v["type"] for v in sub.values()]).value_counts())
# file nộp sạch: đúng thứ tự câu hỏi, mỗi id 1 dòng, thiếu thì ghi abstain
with open(f"{WORK}/submission_public.jsonl", "w", encoding="utf-8") as f:
    for q in qs:
        f.write(json.dumps(sub.get(q, {"id": q, "type": "abstain"}), ensure_ascii=False) + "\n")
print("File nộp:", f"{WORK}/submission_public.jsonl")


In [ ]:
# Nén kết quả + log để tải về
!cd "$WORK" && zip -qr outputs.zip submission_public.jsonl out_public out_train out_smoke -x "*/cache/*" && ls -lh outputs.zip


## Ghi chú
- Chạy lại cùng lệnh sẽ **tiếp tục** từ câu chưa làm (đọc lại `submission.jsonl`); thêm `--fresh` để chạy lại từ đầu.
- Kết quả gọi model được cache ở `cache/llm_cache.jsonl`, đổi prompt thì cache tự khác.
- Phiên Kaggle tối đa 12 giờ: đặt `--time_budget` (giây) để tự dừng an toàn.
- `--n_paths 1` nhanh gấp ~2 lần nhưng không có kiểm tra đồng thuận (precision thấp hơn).